# triton-blackhole — Colab smoke test

1. **Runtime → Change runtime type → GPU (T4)**
2. Run all cells.

Repo: https://github.com/brian-mwirigi/triton-blackhole

**Important:** do not `pip install -U triton`. Colab's torch pins an exact Triton version (e.g. `triton==3.6.0`).

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "Enable a GPU runtime: Runtime → Change runtime type → GPU"

In [ ]:
# Match the triton pin that Colab's torch requires (do NOT upgrade to latest).
import importlib.util
import re
import subprocess
import sys

from importlib.metadata import requires

torch_reqs = requires("torch") or []
pinned = None
for r in torch_reqs:
    m = re.search(r"triton\s*==\s*([0-9][^\s;]+)", r, re.I)
    if m:
        pinned = m.group(1)
        break

if pinned:
    print(f"torch wants triton=={pinned}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", f"triton=={pinned}"])
elif importlib.util.find_spec("triton") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "triton==3.6.0"])

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "git+https://github.com/brian-mwirigi/triton-blackhole.git",
])

import torch
import triton
import triton_blackhole as tb
print("torch", torch.__version__)
print("triton", triton.__version__)
print("triton-blackhole", tb.__version__)

In [ ]:
from triton_blackhole import bisect_axes, classify_drift, compare, format_report
from triton_blackhole.classify import format_classification
import torch

torch.manual_seed(0)
ref = torch.zeros(64, 64, device="cuda")
act = ref.clone()
act[50, 51] = 10
print(format_report(compare(act, ref, atol=1e-6, rtol=1e-6)))
print(format_classification(classify_drift(act, ref, atol=1e-6, rtol=1e-6)))
print(bisect_axes(act, ref, atol=1e-6, rtol=1e-6).report())

In [ ]:
import triton
import triton.language as tl
import torch
from triton_blackhole.triton_hooks import diagnose

@triton.jit
def add_kernel(x_ptr, y_ptr, out_ptr, n, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n
    x = tl.load(x_ptr + offs, mask=mask)
    y = tl.load(y_ptr + offs, mask=mask)
    tl.store(out_ptr + offs, x + y, mask=mask)

def triton_add(x, y):
    out = torch.empty_like(x)
    n = x.numel()
    BLOCK = 256
    add_kernel[(triton.cdiv(n, BLOCK),)](x, y, out, n, BLOCK=BLOCK)
    return out

x = torch.randn(100_000, device="cuda", dtype=torch.bfloat16)
y = torch.randn_like(x)
actual = triton_add(x, y)
reference = x + y
print(diagnose(actual, reference))
print("LIVE TRITON KERNEL OK")